The `api.sqlite` functions work with the `sqlite3` driver from the Python standard library, and need SQLite 3.24 or later. This page shows the behavior that is specific to SQLite, using a database file, as most applications do.

In [1]:
import sqlite3

sqlite3.sqlite_version

'3.50.4'

In [2]:
import tempfile
from pathlib import Path

import sqlalchemy as sa

from sqlalchemy_upsert_kit.api import sqlite

db_path = Path(tempfile.mkdtemp()) / "app.sqlite"
engine = sa.create_engine(f"sqlite:///{db_path}")

SQLite enforces foreign keys only when asked to, once per connection:

In [3]:
@sa.event.listens_for(engine, "connect")
def enable_foreign_keys(dbapi_connection, connection_record):
    dbapi_connection.execute("PRAGMA foreign_keys = ON")

## Rows are updated in place

A parent table, and a child table whose rows are deleted with their parent:

In [4]:
metadata = sa.MetaData()

t_parent = sa.Table(
    "parents",
    metadata,
    sa.Column("id", sa.Integer, primary_key=True),
    sa.Column("name", sa.String(50)),
)

t_child = sa.Table(
    "children",
    metadata,
    sa.Column("id", sa.Integer, primary_key=True),
    sa.Column("parent_id", sa.Integer, sa.ForeignKey("parents.id", ondelete="CASCADE")),
    sa.Column("name", sa.String(50)),
)

metadata.create_all(engine)
with engine.begin() as conn:
    conn.execute(t_parent.insert(), [{"id": 1, "name": "p1"}, {"id": 2, "name": "p2"}])
    conn.execute(t_child.insert(), [{"id": 10, "parent_id": 1, "name": "c10"}, {"id": 11, "parent_id": 2, "name": "c11"}])

And a trigger that logs every update and delete on the parents:

In [5]:
with engine.begin() as conn:
    conn.exec_driver_sql("CREATE TABLE audit (event TEXT)")
    conn.exec_driver_sql(
        "CREATE TRIGGER parents_update AFTER UPDATE ON parents "
        "BEGIN INSERT INTO audit VALUES ('update ' || NEW.id); END"
    )
    conn.exec_driver_sql(
        "CREATE TRIGGER parents_delete AFTER DELETE ON parents "
        "BEGIN INSERT INTO audit VALUES ('delete ' || OLD.id); END"
    )

Replace both parents and add a third:

In [6]:
sqlite.insert_or_replace(
    engine,
    t_parent,
    [{"id": 1, "name": "P1"}, {"id": 2, "name": "P2"}, {"id": 3, "name": "P3"}],
)

(2, 1)

In [7]:
with engine.connect() as conn:
    children = conn.execute(sa.select(t_child).order_by(t_child.c.id)).all()
    audit = conn.exec_driver_sql("SELECT event FROM audit").scalars().all()
print(f"{children = }")
print(f"{audit = }")

children = [(10, 1, 'c10'), (11, 2, 'c11')]
audit = ['update 1', 'update 2']


The children are still there, and the log holds only updates. The conflicting parents were updated, never deleted, so `ON DELETE CASCADE` and delete triggers do not fire. `insert_or_merge` behaves the same way.

## A rollback undoes the upsert, even as the first write

Python's `sqlite3` driver starts a database transaction only at the first statement that changes data. The library makes sure one is open before it starts, so in user-managed mode your rollback undoes the upsert even when the upsert is the first thing you wrote:

In [8]:
with engine.connect() as conn:
    trans = conn.begin()
    sqlite.insert_or_replace(engine, t_parent, [{"id": 1, "name": "rolled back"}], conn=conn, trans=trans)
    trans.rollback()

with engine.connect() as conn:
    name = conn.execute(sa.select(t_parent.c.name).where(t_parent.c.id == 1)).scalar_one()
name

'P1'

## Concurrent writers wait for each other

SQLite allows one writer at a time. Each call takes the database's write lock when it starts, so concurrent calls queue up, each waiting up to the driver's busy timeout (5 seconds by default, the `timeout` argument of `sqlite3.connect`), instead of failing with "database is locked". Four threads, each upserting its own ten parents:

In [9]:
import threading

results = []


def upsert_batch(start):
    batch = []
    for i in range(start, start + 10):
        batch.append({"id": i, "name": f"p{i}"})
    results.append(sqlite.insert_or_ignore(engine, t_parent, batch))


threads = []
for start in [100, 200, 300, 400]:
    threads.append(threading.Thread(target=upsert_batch, args=(start,)))
for thread in threads:
    thread.start()
for thread in threads:
    thread.join()
results

[(0, 10), (0, 10), (0, 10), (0, 10)]

In [10]:
with engine.connect() as conn:
    count = conn.execute(sa.select(sa.func.count()).select_from(t_parent)).scalar_one()
count

43

All 40 rows arrived, next to the 3 parents from before. Each call has its own staging table, private to its connection, so the calls never see each other's data.

## Not supported

- Async engines (`aiosqlite`) and other SQLite drivers.
- A `sqlite3` connection opened with `autocommit=True` (Python 3.12+), or SQLAlchemy's `AUTOCOMMIT` isolation level.